# Seleccion de Caracteristicas Multimetodo
Objetivo: comparar varios metodos de seleccion para `tipo_violencia` y `nivel_riesgo_victima`, y obtener un ranking consensuado de features.

## 1) Librerias

In [ ]:
# !pip install pandas pyarrow numpy scipy scikit-learn matplotlib seaborn
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif, RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 180)
sns.set(style="whitegrid")


## 2) Carga de datos
Carga el parquet final y estandariza nombre de target de riesgo.

In [ ]:
PARQUET_PATH = "/home/munasqa/MAESTRIA/opencode/base_modelado.parquet"
df = pd.read_parquet(PARQUET_PATH)

if "nivel_de_riesgo_victima" in df.columns and "nivel_riesgo_victima" not in df.columns:
    df = df.rename(columns={"nivel_de_riesgo_victima": "nivel_riesgo_victima"})

targets = ["tipo_violencia", "nivel_riesgo_victima"]
for t in targets:
    if t not in df.columns:
        raise ValueError(f"Falta target: {t}")

print(df.shape)
display(df[targets].head())


## 3) Preparacion de features
Excluye targets y posibles fugas evidentes.

In [ ]:
leakage_obvio = ["tipo_violencia", "nivel_riesgo_victima", "tipo_violencia_lbl", "nivel_riesgo_victima_lbl"]
features = [c for c in df.columns if c not in leakage_obvio]

X_raw = df[features].copy()

for c in X_raw.columns:
    if str(X_raw[c].dtype) in ["object", "category"]:
        X_raw[c] = X_raw[c].astype(str).fillna("desconocido")
    else:
        X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")
        X_raw[c] = X_raw[c].fillna(X_raw[c].median())

print("Features candidatas:", len(features))


## 4) Nota metodologica (paper)
El paper de referencia menciona enfoque multi-metodo (filters + wrappers + evolutivo). Aqui implementamos una version practica y reproducible en sklearn:
- Filter: Chi2/Cramers V y Mutual Information
- Embedded: Random Forest importance + Permutation importance
- Wrapper: RFECV
Luego se consolida un ranking por consenso.

## 5) Funciones auxiliares

In [ ]:
def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan, np.nan
    chi2, p, _, _ = chi2_contingency(tab)
    n = tab.values.sum()
    r, k = tab.shape
    den = min(r - 1, k - 1)
    v = np.sqrt((chi2 / n) / den) if den > 0 else np.nan
    return v, p


def to_ordinal_encoded(X_df):
    X = X_df.copy()
    cat_cols = [c for c in X.columns if str(X[c].dtype) in ["object", "category"]]
    num_cols = [c for c in X.columns if c not in cat_cols]

    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    if cat_cols:
        X[cat_cols] = enc.fit_transform(X[cat_cols])

    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(X[c].median())

    return X


def rank_to_score(df_rank, score_col, feature_col="feature", descending=True):
    r = df_rank[[feature_col, score_col]].dropna().copy()
    if r.empty:
        return pd.DataFrame(columns=["feature", "score"])
    r = r.sort_values(score_col, ascending=not descending).reset_index(drop=True)
    r["score"] = np.linspace(1.0, 0.0, len(r), endpoint=False)
    return r[[feature_col, "score"]]


## 6) Seleccion multimetodo por target
Se ejecutan 5 metodos y se combina un score de consenso.

In [ ]:
results = {}
SAMPLE_SIZE = 100000  # Reducir a 100k para evitar OOM (Out of Memory)

for target in targets:
    y_full = pd.to_numeric(df[target], errors="coerce")
    mask = y_full.notna()
    y_full = y_full[mask].astype(int)
    X_full = X_raw.loc[mask]

    # Muestreo estratificado para seleccion de caracteristicas
    if len(y_full) > SAMPLE_SIZE:
        _, X_use, _, y = train_test_split(
            X_full, y_full, 
            test_size=SAMPLE_SIZE, 
            stratify=y_full, 
            random_state=42
        )
    else:
        X_use, y = X_full.copy(), y_full.copy()

    print(f"--- Procesando target: {target} (Muestra: {len(y)} registros) ---")

    # 1) Cramers V (categoricas)
    rows_cv = []
    for c in X_use.columns:
        vals = X_use[c].astype(str)
        v, p = cramers_v(vals, y.astype(str))
        rows_cv.append((c, v, p))
    r_cv = pd.DataFrame(rows_cv, columns=["feature", "cramers_v", "p_value"]).dropna()

    # 2) Mutual Information
    X_enc = to_ordinal_encoded(X_use)
    mi_vals = mutual_info_classif(X_enc, y, discrete_features="auto", random_state=42)
    r_mi = pd.DataFrame({"feature": X_enc.columns, "mi": mi_vals})

    Xtr, Xte, ytr, yte = train_test_split(X_enc, y, test_size=0.2, stratify=y, random_state=42)

    # 3) RF feature_importances_
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample")
    rf.fit(Xtr, ytr)
    r_rf = pd.DataFrame({"feature": X_enc.columns, "rf_importance": rf.feature_importances_})

    # 4) Permutation importance (solo en top 50 de RF para ahorrar RAM/Tiempo)
    top_50_rf = r_rf.sort_values("rf_importance", ascending=False).head(50)["feature"].tolist()
    perm = permutation_importance(rf, Xte, yte, n_repeats=3, random_state=42, n_jobs=-1, scoring="f1_macro")
    r_perm = pd.DataFrame({"feature": X_enc.columns, "perm_importance": perm.importances_mean})

    # 5) RFECV (wrapper)
    # Usamos el top 40 de MI para que el wrapper no sea tan pesado
    top_for_rfe = r_mi.sort_values("mi", ascending=False).head(min(40, len(r_mi)))["feature"].tolist()
    X_rfe = X_enc[top_for_rfe].copy()
    rfe_est = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample")
    rfecv = RFECV(estimator=rfe_est, step=5, cv=3, scoring="f1_macro", min_features_to_select=5, n_jobs=-1)
    rfecv.fit(X_rfe, y)
    r_rfe = pd.DataFrame({"feature": X_rfe.columns, "rfe_rank": rfecv.ranking_, "rfe_selected": rfecv.support_.astype(int)})

    s_cv = rank_to_score(r_cv, "cramers_v", descending=True)
    s_mi = rank_to_score(r_mi, "mi", descending=True)
    s_rf = rank_to_score(r_rf, "rf_importance", descending=True)
    s_perm = rank_to_score(r_perm, "perm_importance", descending=True)

    s_rfe = r_rfe[["feature", "rfe_selected"]].copy()
    s_rfe["score"] = s_rfe["rfe_selected"].astype(float)
    s_rfe = s_rfe[["feature", "score"]]

    pool = pd.DataFrame({"feature": X_enc.columns}).drop_duplicates()
    for name, s in [("cv", s_cv), ("mi", s_mi), ("rf", s_rf), ("perm", s_perm), ("rfe", s_rfe)]:
        pool = pool.merge(s.rename(columns={"score": f"score_{name}"}), on="feature", how="left")

    pool = pool.fillna(0)
    pool["score_consenso"] = (
        0.20 * pool["score_cv"] +
        0.25 * pool["score_mi"] +
        0.25 * pool["score_rf"] +
        0.20 * pool["score_perm"] +
        0.10 * pool["score_rfe"]
    )
    pool = pool.sort_values("score_consenso", ascending=False).reset_index(drop=True)

    yhat = rf.predict(Xte)
    f1m = f1_score(yte, yhat, average="macro")

    results[target] = {
        "cv": r_cv.sort_values("cramers_v", ascending=False),
        "mi": r_mi.sort_values("mi", ascending=False),
        "rf": r_rf.sort_values("rf_importance", ascending=False),
        "perm": r_perm.sort_values("perm_importance", ascending=False),
        "rfe": r_rfe.sort_values(["rfe_selected", "rfe_rank"], ascending=[False, True]),
        "consenso": pool,
        "rf_f1_macro_test": f1m,
    }

    print(f"Target: {target} | RF (muestra) f1_macro test: {f1m:.4f}")
    display(pool.head(20))
